# Lista 9

## Tensorboard and WandB

(5pkt + 2pkt)

Na liście znajduje się 1 zadanie. Po rozwiązaniu go, pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Dodatkowe zadanie oznaczone jest ⭐️ i jest warte 2pkt.

## Jeśli ćwiczenia będą się przedłuały...

Odpowiedz na ponisze pytania pisemnie i prześlij zadanie na skos. Do zobaczenia na kolejnych zajęciach! 😀

1. Otwórz Projector i uruchom jeden ze sposobów wizualizacji danych. Napisz który uruchomiłeś. Przeanalizuj embeddingi i naapisz co ciekawego zobaczyłeś.
2. Która klasa na wykresie PR courve wypadła najgorzej? Po czym to poznałeś/poznałaś?
3. Otwórz zakładkę, w której zobaczysz graf sieci neuronowej. W jakie fragmenty sieci musisz "kliknąć" w celu zobaczenia warstwy conv2?
4. (Nieobowiązkowe dla obecnych, obowiązkowe dla nieobecnych) Dodaj tqdm do treningu sieci.

# TensorBoard 

## **1. Wczytanie danych i przygotowanie transformacji**

1. Wczytaj zbiór **CIFAR-10** (`torchvision.datasets.CIFAR10`).
2. Zastosuj transformacje:

   * `ToTensor()`
   * normalizacja kanałów:
     `mean = (0.5, 0.5, 0.5)`
     `std  = (0.5, 0.5, 0.5)`
3. Przygotuj **DataLoader** dla train i test z `batch_size=4`.

---

## **2. Uruchomienie TensorBoard + zapis przykładowych obrazów**

1. Zainicjalizuj logger:
   `writer = SummaryWriter('runs/cifar10_experiment')`
2. Pobierz jeden batch z trainloadera i:

   * stwórz grid (`make_grid`)
   * zapisz go przez `writer.add_image(...)`.

---

## **3. Wizualizacja architektury modelu**

1. Zaimplementuj prostą CNN (2×Conv2d, 3×Linear).
2. Użyj przykładowego batcha z DataLoadera i zapisz graf:
   `writer.add_graph(model, images)`.

---

## **4. Embedding – 100 losowych obrazów**

1. Wybierz 100 losowych przykładów z train set (numpy → tensor).
2. Zamień format (N, H, W, C) → (N, C, H, W).
3. Spłaszcz każdy obraz do wymiaru **3072**.
4. Zapisz embedding (`add_embedding`):

   * `metadata = nazwy klas`,
   * `label_img = obrazki`.

---

## **5. Śledzenie treningu**

1. Przeprowadź jedną epokę treningu.
2. Co 500 iteracji:

   * `writer.add_scalar('training_loss', loss)`
   * stwórz figure Matplotlib z predykcjami vs. prawdziwe etykiety i zapisz przez `add_figure`.

---

## **6. Ocena modelu – Precision-Recall**

1. Przelicz softmax na całym zbiorze testowym.
2. Dla każdej klasy (0–9):

   * przygotuj maskę prawdy i wektor prawdopodobieństw,
   * zapisz wykres PR: `writer.add_pr_curve(...)`.

---

## **7. Zakończenie**

Na końcu wywołaj `writer.close()`.
TensorBoard uruchom komendą:

```bash
tensorboard --logdir=runs
```

In [7]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np
from torch.utils.tensorboard import SummaryWriter

# ================================================================
# 1. WCZYTANIE DANYCH + TRANSFORMACJE
# ================================================================

# Transformacje: konwersja do tensora + normalizacja kanałów RGB
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Wczytanie zbioru CIFAR-10 (train i test)
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# DataLoadery – batch_size = 4 (dla przejrzystości w TensorBoard)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=4, shuffle=True
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=4, shuffle=False
)

classes = trainset.classes  # nazwy klas CIFAR-10

# ================================================================
# Helper: wyświetlanie obrazów w figure Matplotlib
# ================================================================
def matplotlib_imshow(img):
    # "odwrócenie" normalizacji: [-1,1] -> [0,1]
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1,2,0)))

# ================================================================
# 2. TWORZENIE I URUCHAMIANIE TENSORBOARD + ZAPIS PIERWSZYCH OBRAZÓW
# ================================================================

# Inicjalizacja loggera TensorBoard
writer = SummaryWriter('runs/cifar10_experiment')

# Pobranie jednego batcha i zapis jako obraz (grid)
dataiter = iter(trainloader)
images, labels = next(dataiter)
img_grid = torchvision.utils.make_grid(images)

# Zapis obrazów do TensorBoard (TensorBoard → Images)
writer.add_image('image', img_grid)

# ================================================================
# 3. DEFINICJA I WIZUALIZACJA ARCHITEKTURY SIECI
# ================================================================

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Prosta CNN: 2 warstwy konwolucyjne + 3 w pełni połączone
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 1. Conv + ReLU + Pool
        x = self.pool(F.relu(self.conv2(x)))  # 2. Conv + ReLU + Pool
        x = x.view(-1, 16*5*5)                # spłaszczenie
        x = F.relu(self.fc1(x))               # FC1
        x = F.relu(self.fc2(x))               # FC2
        x = self.fc3(x)                       # FC3 (wyjście logits)
        return x

net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# Zapis grafu modelu do TensorBoard
# (TensorBoard → Graph)
writer.add_graph(net, images)

# ================================================================
# 4. EMBEDDING 100 LOSOWYCH OBRAZKÓW
# ================================================================

# Helper: wybór losowych obrazków i ich etykiet
def select_n_random(data, labels, n=100):
    assert len(data) == len(labels)
    perm = torch.randperm(len(data))
    return data[perm][:n], torch.tensor(labels)[perm][:n]

# Wybór 100 elementów
images, labels = select_n_random(trainset.data, trainset.targets)

# Zamiana z numpy (N,H,W,C) → tensor (N,C,H,W)
images = torch.tensor(images).permute(0, 3, 1, 2).float()

# Zamiana etykiet na nazwy klas
class_labels = [classes[l] for l in labels]

# Spłaszczenie obrazów do wymiaru 3072 (32×32×3)
features = images.view(100, -1)

# Zapis do TensorBoard (→ zakładka Projector)
# Uzupełnij (add_embedding, label_img=images/255.0)
writer.add_embedding(features, metadata=class_labels, label_img=images/255.0)

# ================================================================
# 5. ŚLEDZENIE TRENINGU (SCALARS + IMAGES)
# ================================================================

# Helper: obliczanie predykcji + prawdopodobieństw
def images_to_probs(net, images):
    output = net(images)
    _, preds = torch.max(output, 1)  # predykcje
    # wyciągnięcie prawdopodobieństwa każdej przewidzianej klasy
    probs = [F.softmax(o, dim=0)[p].item() for o,p in zip(output, preds)]
    return preds, probs

# Helper: tworzenie figure z predykcjami vs. etykiety
def plot_classes_preds(net, images, labels):
    preds, probs = images_to_probs(net, images)
    fig = plt.figure(figsize=(12,4))

    for i in range(4):
        ax = fig.add_subplot(1,4,i+1)
        matplotlib_imshow(images[i])
        ax.set_title(
            f"{classes[preds[i]]}: {probs[i]*100:.1f}%\n"
            f"(label: {classes[labels[i]]})",
            color=("green" if preds[i] == labels[i] else "red")
        )
        ax.axis('off')
    return fig

# ------------------------------
# Pętla treningowa: 1 epoka
# co 500 batchy zapis do TensorBoard
# ------------------------------
running_loss = 0.0
for epoch in range(5):
    for i, (inputs, labels) in enumerate(trainloader):

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # zapis co 500 iteracji
        if i % 500 == 499:
            # Zapis training loss
            # Uzupełnij (add_scalar)
            writer.add_scalar('training_loss', loss, global_step=epoch * (len(trainloader)) + i)

            # Images → predykcje vs rzeczywiste etykiety
            fig = plot_classes_preds(net, inputs, labels)
            # Uzupełnij (add_figure)
            writer.add_figure('figure', fig)

            running_loss = 0.0

# ================================================================
# 6. KRZYWE PRECISION–RECALL DLA 10 KLAS
# ================================================================

class_probs = []
class_label = []

# Zebranie predykcji na danych testowych
with torch.no_grad():
    for images, labels in testloader:
        outputs = net(images)
        probs = F.softmax(outputs, dim=1)
        class_probs.append(probs)
        class_label.append(labels)

test_probs = torch.cat(class_probs)   # macierz [N,10]
test_label = torch.cat(class_label)   # wektor [N]

# Funkcja zapisująca PR curve jednej klasy
def add_pr_curve(class_index):
    truth = (test_label == class_index)     # maska prawdziwych przykładów
    probs = test_probs[:, class_index]      # prawdopodobieństwa tej klasy

    # PR curve do TensorBoard
    writer.add_pr_curve(classes[class_index], truth, probs)


# PR curves dla wszystkich klas CIFAR-10
for i in range(10):
    add_pr_curve(i)

# Zamykamy logger
writer.close()

Files already downloaded and verified
Files already downloaded and verified


# ⭐️ Weights & Biases

Twoim zadaniem jest stworzenie **mini-eksperymentu machine learningowego**, który będzie w pełni rejestrowany w systemie **Weights & Biases (W&B)**.

---

### 1️ Zainstaluj i połącz się z W&B

Użyj `wandb.login()`, aby połączyć się ze swoim kontem.

---

### 2️ Przygotuj słownik hiperparametrów

Umieść tam m.in.:

* liczbę epok,
* batch size,
* learning rate,
* informację o architekturze i zbiorze danych.

To wszystko zostanie automatycznie zarejestrowane w W&B.

---

### 3️ Zaimplementuj pipeline treningowy

Napisz funkcję `model_pipeline(config)`, która:

* **inicjalizuje eksperyment** w W&B,
* tworzy dane, model, loss i optimizer,
* trenuje model i **loguje metryki**,
* testuje go na zbiorze testowym,
* **zapisuje model** do formatu ONNX i wrzuca go do W&B.

---

### 4️ Loguj różne rzeczy do W&B

Przynajmniej:

* stratę podczas treningu (`loss`),
* numer epoki,
* parametry i gradienty modelu (poprzez `run.watch()`).
* dokładność na danych testowych (po treningu)

---

### 5 Uruchom eksperyment i odwiedź stronę W&B

Sprawdź:

* wykres strat,
* gradienty i parametry,
* zapisany model,
* podsumowanie runu.

In [1]:
# ================================
# 1. Importy i przygotowanie środowiska
# ================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm
import wandb

# Ustalanie stałych seedów, aby wyniki były powtarzalne
torch.backends.cudnn.deterministic = True
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

# Automatyczny wybór GPU, jeśli dostępne
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Logowanie do platformy Weights & Biases
# Uzupełnij
wandb.login()


/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: (__ZN3c1017RegisterOperatorsD1Ev)
  Referenced from: '/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so'
  Expected in: '/opt/miniconda3/envs/pytorch_lab/lib/libtorch_cpu.dylib''If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
wandb: Currently logged in as: kornel-orawczak (kornel-orawczak-uniwersytet-wroc-awski) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:

# ================================
# 2. Konfiguracja hiperparametrów
# ================================

# Uzupełnij (config:
# epochs=3,                # liczba epok
# batch_size=128,          # rozmiar batcha
# learning_rate=0.001,     # learning rate
# dataset="MNIST",         # używany zbiór danych
# architecture="SimpleCNN",# meta-info o modelu
# classes=10,              # liczba klas w MNIST
# kernels=[16, 32]         # liczba filtrów w warstwach CNN)
config = dict(
    epochs=3,
    batch_size=128,
    learning_rate=0.001,
    dataset="MNIST",
    architecture="SimpleCNN",
    classes=10, 
    kernels=[16, 32] 
)


# ================================
# 3. Funkcje pomocnicze — dane
# ================================

def get_data(train=True):
    """
    Pobiera zbiór MNIST i konwertuje go na tensory.
    """
    dataset = torchvision.datasets.MNIST(
        root=".",
        train=train,
        transform=transforms.ToTensor(),
        download=True
    )
    return dataset


def make_loader(dataset, batch_size):
    """
    Tworzy DataLoader dla MNIST.
    """
    return torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )


# ================================
# 4. Definicja modelu CNN
# ================================

class ConvNet(nn.Module):
    """
    Prosta, dwuwarstwowa sieć konwolucyjna + warstwa w pełni połączona.
    """
    def __init__(self, kernels, classes=10):
        super().__init__()

        # Pierwsza warstwa konwolucyjna
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, kernels[0], kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Druga warstwa konwolucyjna
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernels[0], kernels[1], kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Warstwa w pełni połączona
        self.fc = nn.Linear(7 * 7 * kernels[1], classes)

    def forward(self, x):
        """
        Definicja przepływu danych w sieci.
        """
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)


# ================================
# 5. Funkcja treningowa — logowanie W&B
# ================================

def train(model, loader, criterion, optimizer, config, run):
    """
    Trenuje model i loguje metryki do W&B.
    """

    # Automatyczne logowanie gradientów i wag modelu
    # Uzupełnij (watch)
    wandb.watch(model, criterion, log='all', log_freq=10)
    example_ct = 0
    for epoch in range(config.epochs):
        for batch, (images, labels) in enumerate(loader):

            # Przeniesienie danych na GPU (jeśli dostępne)
            images, labels = images.to(device), labels.to(device)

            # Zerowanie gradientów
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Aktualizacja wag
            optimizer.step()
            example_ct += len(images)

            # Logowanie strat co 50 batchy
            if batch % 50 == 0:
                # Uzupełnij (log: loss, epoch)
                wandb.log({"epoch": epoch, "loss": loss}, step=example_ct)

# ================================
# 6. Funkcja testowa + zapis modelu ONNX
# ================================

def test(model, loader, run):
    """
    Testuje model oraz zapisuje jego parametry na W&B.
    """

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            # Statystyki dokładności
            total += labels.size(0)
            correct += (pred == labels).sum().item()

    accuracy = correct / total

    # Logowanie dokładności testowej
    # Uzupełnij (log: accuracy)
    wandb.log({"test_accuracy": accuracy})
    print("Test accuracy:", accuracy)

    # Eksport modelu do ONNX
    dummy_input = torch.randn(1, 1, 28, 28).to(device)
    # Uzupełnij (exportowanie modelu do formatu ONNX)
    torch.onnx.export(model, images, "model.onnx")
    # Zapisanie modelu w W&B
    # Uzupełnij (zapis modelu do W&B)
    wandb.save("model.onnx")


# ================================
# 7. Pełny pipeline eksperymentu
# ================================

def model_pipeline(hyperparameters):
    """
    Główna funkcja kontrolująca cały proces:
    - inicjalizacja W&B
    - trening
    - testowanie
    - zapis modelu
    """

    # Inicjalizacja eksperymentu
    # Uzupełnij (wband init... ) as run:
    with wandb.init(project="list9-demo", config=hyperparameters) as run:
        config = run.config  # Wersja configu synchronizowana z W&B

        # 1. Przygotowanie danych
        train_dataset = get_data(train=True)
        test_dataset = get_data(train=False)
        train_loader = make_loader(train_dataset, config.batch_size)
        test_loader = make_loader(test_dataset, config.batch_size)

        # 2. Utworzenie modelu
        model = ConvNet(config.kernels, config.classes).to(device)

        # 3. Loss + optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

        # 4. Trening
        train(model, train_loader, criterion, optimizer, config, run)

        # 5. Testowanie + zapis modelu
        test(model, test_loader, run)

    return model


# ================================
# 8. Uruchomienie eksperymentu
# ================================

model = model_pipeline(config)

/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: (__ZN3c1017RegisterOperatorsD1Ev)
  Referenced from: '/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so'
  Expected in: '/opt/miniconda3/envs/pytorch_lab/lib/libtorch_cpu.dylib''If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: (__ZN3c

Test accuracy: 0.9847


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch,▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅██████████
loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
epoch,2
loss,0.08423
test_accuracy,0.9847
